In [1]:
import os
import pickle

# =========================
# 0) AIOps 固定调用拓扑（与原 TopologyDao 一致）
# =========================
DEFAULT_CALL_RELATIONS = [
    "frontend -> adservice",
    "frontend -> cartservice",
    "frontend -> checkoutservice",
    "frontend -> currencyservice",
    "frontend -> recommendationservice",
    "frontend -> productcatalogservice",
    "frontend -> shippingservice",
    "checkoutservice -> cartservice",
    "checkoutservice -> currencyservice",
    "checkoutservice -> emailservice",
    "checkoutservice -> paymentservice",
    "checkoutservice -> productcatalogservice",
    "checkoutservice -> shippingservice",
    "recommendationservice -> productcatalogservice",
]

# 与原代码一致：service 对应 4 个 pod 名后缀
DEFAULT_POD_SUFFIXES = ["-0", "-1", "-2", "2-0"]


def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)
    return p


# =========================
# 1) 构图：返回 edge_index = [src_list, dst_list]
# =========================
def build_aiops_topology_edge_index(
    all_entity_list: list,
    service_order_list: list,
    call_relations=DEFAULT_CALL_RELATIONS,
    pod_suffixes=DEFAULT_POD_SUFFIXES,
    add_self_loops: bool = True,
):
    """
    edge_index 格式： [ [src0, src1, ...], [dst0, dst1, ...] ]
    与 PyG 常见 edge_index 兼容（COO）
    """
    name2idx = {name: i for i, name in enumerate(all_entity_list)}
    src, dst = [], []

    # 1) service <-> pod
    for service in service_order_list:
        if service not in name2idx:
            raise ValueError(f"[topology] service not in all_entity_list: {service}")
        s_idx = name2idx[service]

        for suf in pod_suffixes:
            pod = f"{service}{suf}"

            p_idx = name2idx[pod]

            # pod -> service
            src.append(p_idx); dst.append(s_idx)
            # service -> pod
            src.append(s_idx); dst.append(p_idx)

    # 2) service <-> service call graph
    for rel in call_relations:
        caller, callee = rel.split(" -> ")
        if caller not in name2idx or callee not in name2idx:
            raise ValueError(f"[topology] caller/callee missing in all_entity_list: {rel}")

        c1, c2 = name2idx[caller], name2idx[callee]
        # caller -> callee
        src.append(c1); dst.append(c2)
        # callee -> caller
        src.append(c2); dst.append(c1)

    # 3) self loops
    if add_self_loops:
        for i in range(len(all_entity_list)):
            src.append(i); dst.append(i)

    return [src, dst]


# =========================
# 2) 枚举 AIOps 数据目录：dataset_type/date/cloudbed
#    datasets/<dataset_type>/<date>/<cloudbed>/...
# =========================
def iter_aiops_cases(data_base: str):
    for dataset_type in sorted(os.listdir(data_base)):
        ds_dir = os.path.join(data_base, dataset_type)
        if not os.path.isdir(ds_dir):
            continue

        for date in sorted(os.listdir(ds_dir)):
            date_dir = os.path.join(ds_dir, date)
            if not os.path.isdir(date_dir):
                continue

            for cloudbed in sorted(os.listdir(date_dir)):
                cb_dir = os.path.join(date_dir, cloudbed)
                if not os.path.isdir(cb_dir):
                    continue
                yield dataset_type, date, cloudbed


# =========================
# 3) 总控：生成并按路径写出 edge_index.pkl
# =========================
def generate_aiops_topology(
    data_base: str,
    out_base: str,
    all_entity_list: list,
    service_order_list: list,
    skip_bad_case: bool = True,
):
    edge_index = build_aiops_topology_edge_index(
        all_entity_list=all_entity_list,
        service_order_list=service_order_list,
    )

    for dataset_type, date, cloudbed in iter_aiops_cases(data_base):
        # 保持与你之前一致的坏例跳过逻辑
        if skip_bad_case and date == "2022-03-24" and cloudbed in ["cloudbed-1", "cloudbed-2"]:
            continue

        out_dir = ensure_dir(os.path.join(out_base, dataset_type, date, cloudbed, "resource_entity"))
        out_pkl = os.path.join(out_dir, "edge_index.pkl")

        with open(out_pkl, "wb") as f:
            pickle.dump(edge_index, f)

        print(f"[OK] saved topology -> {out_pkl}")

In [2]:
all_entity_list = [
    'node-1', 
    'node-2', 
    'node-3', 
    'node-4', 
    'node-5', 
    'node-6', 
    'adservice', 
    'cartservice', 
    'checkoutservice', 
    'currencyservice', 
    'emailservice', 
    'frontend', 
    'paymentservice', 
    'productcatalogservice', 
    'recommendationservice', 
    'shippingservice', 
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [3]:
service_order_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

In [4]:


generate_aiops_topology(
    data_base="/home/ZZData/Eadro/preprocess/datasets/aiops2022/",
    out_base="../../temp_data/aiops22/raw_data",
    all_entity_list=all_entity_list,
    service_order_list=service_order_list,
)

[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-01/cloudbed/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-03/cloudbed/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-05/cloudbed/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-07/cloudbed/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-09/cloudbed/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/groundtruth/.ipynb_checkpoints/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/training_data_normal/2022-03-19/.ipynb_checkpoints/resource_entity/edge_index.pkl
[OK] saved topology -> ../../temp_data/aiops22/raw_data/training_data_normal/2022-03-19/cloudbed-1/resource_entity/edge_index.pkl
[OK] saved topology